# Data Science & Machine Learning — Atelier 1 V2
## Import, inspection, qualité des données et EDA

### Objectifs
À la fin de cet atelier, vous devez être capables de :
- charger et inspecter un jeu de données ;
- distinguer observations, variables et types de données ;
- identifier les valeurs manquantes et discuter leur origine ;
- **justifier** une stratégie de traitement : moyenne, médiane, mode, suppression ou autre ;
- observer distributions, outliers et relations entre variables ;
- construire une première analyse exploratoire des données (EDA) ;
- comprendre ce qu'apporte un rapport EDA automatisé.

> **Fil conducteur : avant de construire un modèle, il faut comprendre les données.**

## ÉTAPE 1 — Import & inspection

### 1.1 Importer les bibliothèques

- **Pandas** : manipulation et analyse de données.
- **NumPy** : calcul numérique et tableaux.

In [ ]:
import pandas as pd
import numpy as np

### 1.2 Charger le dataset

Dans Google Colab, adaptez le chemin si nécessaire.

> Une ligne = une **observation**  
> Une colonne = une **variable**

In [ ]:
DATA_PATH = "/content/drive/MyDrive/dataset_energie_batiments_pedagogique.csv"
df = pd.read_csv(DATA_PATH)

df.head()

### 1.3 Vérifier les dimensions

In [ ]:
print("Nombre d'observations :", df.shape[0])
print("Nombre de variables :", df.shape[1])

### 1.4 Examiner la structure

`info()` permet notamment de repérer :
- les types de colonnes ;
- les colonnes incomplètes ;
- d'éventuelles incohérences de typage.

In [ ]:
df.info()

### 1.5 Statistiques descriptives

`describe()` donne notamment :
- `count` : nombre de valeurs ;
- `mean` : moyenne ;
- `std` : écart-type ;
- quartiles ;
- minimum et maximum.

In [ ]:
df.describe(include='all').T

### 1.6 Uniformiser les noms de colonnes

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df.columns.tolist()

### Question — Que sait-on déjà ?

Prenez 2 minutes pour répondre :
1. Combien avons-nous d'observations et de variables ?
2. Quelles variables sont numériques ? catégorielles ?
3. Quelles colonnes semblent incomplètes ?
4. Voyez-vous déjà des valeurs qui mériteraient d'être vérifiées ?

## ÉTAPE 2 — Valeurs manquantes & qualité des données

### 2.1 Détecter les valeurs manquantes

In [ ]:
missing = pd.DataFrame({
    "nb_manquantes": df.isnull().sum(),
    "pourcentage": (df.isnull().mean() * 100).round(2)
})

missing[missing["nb_manquantes"] > 0].sort_values("pourcentage", ascending=False)

### Question essentielle : pourquoi la valeur manque-t-elle ?

Une valeur manquante n'implique pas automatiquement qu'il faut la remplacer.

Selon le contexte, on peut :
- supprimer quelques observations ;
- supprimer une variable très incomplète et peu utile ;
- imputer une valeur ;
- créer une catégorie `Inconnu` ;
- conserver un indicateur signalant que la valeur était manquante.

> **Le traitement doit être justifié, pas automatique.**

### 2.2 Observer les variables numériques concernées

In [ ]:
cols_num = [
    c for c in ["surface_m2", "energy_consumption_kwh", "co2_emissions_kg"]
    if c in df.columns
]

df[cols_num].describe().T

### 2.3 Moyenne ou médiane ?

**Moyenne** : pertinente lorsque la distribution est assez symétrique et sans valeurs extrêmes majeures.

**Médiane** : souvent préférable lorsque la distribution est asymétrique ou contient des outliers, car elle est moins sensible aux valeurs extrêmes.

Pour une variable catégorielle, on peut utiliser le **mode** ou une catégorie dédiée comme `Inconnu`.

**Questions à se poser :**
1. La distribution est-elle symétrique ?
2. Y a-t-il des valeurs extrêmes ?
3. Quelle proportion des valeurs manque ?
4. L'absence elle-même peut-elle porter une information ?

In [ ]:
# Comparaison moyenne / médiane sur les variables numériques concernées
comparaison = pd.DataFrame({
    "moyenne": df[cols_num].mean(),
    "mediane": df[cols_num].median(),
    "ecart_moyenne_mediane": (df[cols_num].mean() - df[cols_num].median()).abs()
})

comparaison

### 2.4 Visualiser AVANT d'imputer

La visualisation aide à justifier le choix de l'imputation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

for col in cols_num:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[col], bins=30, kde=True)
    plt.axvline(df[col].mean(), linestyle="--", label="Moyenne")
    plt.axvline(df[col].median(), linestyle=":", label="Médiane")
    plt.title(f"Distribution de {col}")
    plt.legend()
    plt.show()

### 2.5 Imputation pour l'EDA

Pour **cet atelier exploratoire**, nous choisissons ici la médiane pour les variables numériques incomplètes.

⚠️ **Important pour le Machine Learning :** dans l'Atelier 2, nous repartirons du dataset brut et l'imputation sera apprise **uniquement sur le jeu d'entraînement**, à l'intérieur du pipeline. Cela évite qu'une information provenant du jeu de test participe à la préparation du modèle.

In [ ]:
df_eda = df.copy()

for col in cols_num:
    df_eda[col] = df_eda[col].fillna(df_eda[col].median())

df_eda.isnull().sum()

### À compléter

> « Nous avons choisi ici d'imputer les valeurs numériques par la médiane parce que … »

Puis indiquez une situation dans laquelle vous auriez préféré :
- la moyenne ;
- la suppression ;
- une catégorie `Inconnu`.

## ÉTAPE 3 — Distributions & visualisations

### 3.1 Distribution de la surface

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df_eda["surface_m2"], bins=30, kde=True)
plt.title("Distribution des surfaces des bâtiments")
plt.xlabel("Surface (m²)")
plt.ylabel("Nombre de bâtiments")
plt.show()

### 3.2 Distribution de la consommation énergétique

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df_eda["energy_consumption_kwh"], bins=30, kde=True)
plt.title("Distribution de la consommation énergétique")
plt.xlabel("Consommation énergétique")
plt.ylabel("Nombre de bâtiments")
plt.show()

### 3.3 Boxplots — repérer dispersion et valeurs atypiques

Un boxplot permet de visualiser :
- la médiane ;
- les quartiles ;
- l'étendue centrale des données ;
- les valeurs potentiellement atypiques.

> Une valeur atypique n'est pas automatiquement une erreur : elle doit être interprétée.

In [ ]:
for col, titre in [
    ("surface_m2", "Surface des bâtiments"),
    ("energy_consumption_kwh", "Consommation énergétique")
]:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df_eda[col])
    plt.title(f"Boxplot — {titre}")
    plt.show()

### 3.4 Relation surface / consommation

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(
    x="surface_m2",
    y="energy_consumption_kwh",
    data=df_eda,
    alpha=0.6
)
plt.title("Surface vs consommation énergétique")
plt.show()

### 3.5 Ajouter une variable catégorielle

In [ ]:
if "building_type" in df_eda.columns:
    plt.figure(figsize=(8,6))
    sns.scatterplot(
        x="surface_m2",
        y="energy_consumption_kwh",
        hue="building_type",
        data=df_eda,
        alpha=0.7
    )
    plt.title("Surface vs consommation selon le type de bâtiment")
    plt.show()

### 3.6 Corrélations numériques

In [ ]:
numeric_cols = df_eda.select_dtypes(include=np.number).columns

plt.figure(figsize=(9,7))
sns.heatmap(
    df_eda[numeric_cols].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Matrice de corrélation")
plt.show()

### Questions d'interprétation

1. Quelles distributions sont symétriques ou asymétriques ?
2. Quels outliers observez-vous ?
3. La surface semble-t-elle liée à la consommation ?
4. Certaines variables sont-elles corrélées ?
5. **Corrélation signifie-t-elle causalité ?** Non.
6. Quelles hypothèses pourriez-vous formuler avant de construire un modèle ?

## ÉTAPE 4 — Rapport EDA automatisé

Un rapport EDA automatisé synthétise rapidement :
- types de variables ;
- valeurs manquantes ;
- distributions ;
- statistiques descriptives ;
- corrélations ;
- alertes sur certaines anomalies.

Il **complète** l'analyse manuelle mais ne remplace pas l'interprétation du data scientist.

In [ ]:
!pip -q install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df_eda,
    title="Rapport d'exploration — Énergie & Bâtiments",
    explorative=True
)

profile.to_notebook_iframe()

## ÉTAPE 5 — Synthèse de l'Atelier 1

À ce stade, vous devez être capables de répondre :

- Que contient le dataset ?
- Quelles sont les principales limites de qualité ?
- Comment avez-vous traité les valeurs manquantes et pourquoi ?
- Quelles distributions ou valeurs atypiques avez-vous observées ?
- Quelles relations semblent intéressantes ?
- Quelles hypothèses pourriez-vous tester en Machine Learning ?

### À retenir

> **EDA = comprendre avant de modéliser.**

Dans l'Atelier 2, nous repartirons volontairement du **dataset brut** pour apprendre à traiter les valeurs manquantes correctement dans un pipeline ML.

### Option — sauvegarder une version nettoyée pour l'analyse descriptive

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/dataset_energie_batiments_nettoye_atelier1.csv"
df_eda.to_csv(OUTPUT_PATH, index=False)
print("Fichier sauvegardé :", OUTPUT_PATH)